In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

cleaned_path = (
    PROCESSED_DATA_DIR
    / "dft_central_spending_2025_10_to_2026_03_cleaned.csv"
)

df = pd.read_csv(
    cleaned_path,
    dtype={
        "transaction_no": "string",
        "postal_code": "string"
    },
    parse_dates=["payment_date"]
)

print("Records loaded:", len(df))
print("Reporting months:", sorted(df["payment_month"].unique()))
print("Missing payment amounts:", df["payment_amount_gbp"].isna().sum())

Records loaded: 3845
Reporting months: ['2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03']
Missing payment amounts: 0


In [2]:
database_path = PROCESSED_DATA_DIR / "dft_spending.sqlite"

conn = sqlite3.connect(database_path)

df.to_sql(
    "payments",
    conn,
    if_exists="replace",
    index=False
)

check = pd.read_sql_query(
    "SELECT COUNT(*) AS records_in_database FROM payments;",
    conn
)

print("Database:", database_path)
display(check)

Database: /Users/dafyddjones/projects/finance-kpi-reporting-pipeline/data/processed/dft_spending.sqlite


,records_in_database
0,3845


In [3]:
monthly_sql = """
SELECT
    payment_month,
    COUNT(*) AS payment_records,
    COUNT(DISTINCT supplier) AS distinct_supplier_names,
    ROUND(SUM(payment_amount_gbp), 2) AS published_payments_gbp
FROM payments
GROUP BY payment_month
ORDER BY payment_month;
"""

sql_monthly_summary = pd.read_sql_query(monthly_sql, conn)

display(
    sql_monthly_summary.style.format({
        "payment_records": "{:,.0f}",
        "distinct_supplier_names": "{:,.0f}",
        "published_payments_gbp": "£{:,.2f}"
    })
)

,payment_month,payment_records,distinct_supplier_names,published_payments_gbp
0,2025-10,711,352,"£3,123,556,273.58"
1,2025-11,531,275,"£2,366,997,852.94"
2,2025-12,543,283,"£2,900,228,687.01"
3,2026-01,659,336,"£2,391,570,289.87"
4,2026-02,606,298,"£2,624,735,301.42"
5,2026-03,795,424,"£3,202,766,377.35"


In [4]:
supplier_sql = """
SELECT
    supplier,
    COUNT(*) AS payment_records,
    ROUND(SUM(payment_amount_gbp), 2) AS published_payments_gbp,
    ROUND(
        100.0 * SUM(payment_amount_gbp)
        / (SELECT SUM(payment_amount_gbp) FROM payments),
        2
    ) AS share_of_total_pct
FROM payments
GROUP BY supplier
ORDER BY published_payments_gbp DESC
LIMIT 10;
"""

top_suppliers = pd.read_sql_query(supplier_sql, conn)

display(
    top_suppliers.style.format({
        "payment_records": "{:,.0f}",
        "published_payments_gbp": "£{:,.2f}",
        "share_of_total_pct": "{:.2f}%"
    })
)

,supplier,payment_records,published_payments_gbp,share_of_total_pct
0,Network Rail,27,"£5,650,598,206.00",34.02%
1,National Highways Limited,7,"£2,441,027,403.19",14.70%
2,GOVIA THAMESLINK RAILWAY LIMITED,23,"£1,021,007,836.40",6.15%
3,GREAT WESTERN RAILWAY,29,"£987,849,628.42",5.95%
4,NORTHERN TRAINS LIMITED,13,"£690,955,934.50",4.16%
5,LONDON NORTH EASTERN RAILWAY,11,"£456,154,755.50",2.75%
6,TRANSPORT UK EAST MIDLANDS LIMITED,21,"£308,193,881.29",1.86%
7,FIRST TRENITALIA RAIL,19,"£283,303,259.95",1.71%
8,WEST MIDLANDS TRAINS,42,"£276,521,550.99",1.66%
9,SE Trains Ltd,17,"£200,902,126.46",1.21%


In [5]:
category_sql = """
SELECT
    expense_type,
    COUNT(*) AS payment_records,
    COUNT(DISTINCT supplier) AS distinct_supplier_names,
    ROUND(SUM(payment_amount_gbp), 2) AS published_payments_gbp,
    ROUND(
        100.0 * SUM(payment_amount_gbp)
        / (SELECT SUM(payment_amount_gbp) FROM payments),
        2
    ) AS share_of_total_pct
FROM payments
GROUP BY expense_type
ORDER BY published_payments_gbp DESC
LIMIT 10;
"""

top_categories = pd.read_sql_query(category_sql, conn)

display(
    top_categories.style.format({
        "payment_records": "{:,.0f}",
        "distinct_supplier_names": "{:,.0f}",
        "published_payments_gbp": "£{:,.2f}",
        "share_of_total_pct": "{:.2f}%"
    })
)

,expense_type,payment_records,distinct_supplier_names,published_payments_gbp,share_of_total_pct
0,Grt Aid to NDPBs,32,4,"£5,505,642,352.00",33.15%
1,TOCOpCosts(Pub),154,18,"£4,844,274,834.80",29.17%
2,Cap Grt Loc Auth,619,143,"£1,999,072,987.53",12.04%
3,CurrGrtOth to NDPBs,47,5,"£1,687,552,989.86",10.16%
4,Cap Grt NDPBs & CG,24,5,"£957,378,383.12",5.76%
5,Cap Grt Pri Sec-Cos.,248,56,"£203,497,184.81",1.23%
6,Subsidies Private Se,944,131,"£185,114,233.54",1.11%
7,Eurotunnel Admin Pay,24,3,"£181,531,582.21",1.09%
8,Curr Grt Other CG Bo,12,3,"£164,456,237.12",0.99%
9,CurrGrt Las in AEF/A,198,98,"£141,265,984.29",0.85%


In [6]:
category_monthly_sql = """
SELECT
    payment_month,
    expense_type,
    COUNT(*) AS payment_records,
    ROUND(SUM(payment_amount_gbp), 2) AS published_payments_gbp
FROM payments
GROUP BY payment_month, expense_type
ORDER BY payment_month, published_payments_gbp DESC;
"""

category_monthly = pd.read_sql_query(
    category_monthly_sql, conn
)

category_trends = category_monthly.pivot(
    index="expense_type",
    columns="payment_month",
    values="published_payments_gbp"
).fillna(0)

top_five_categories = top_categories["expense_type"].head(5)

display(
    category_trends.loc[top_five_categories]
    .style.format("£{:,.0f}")
)

payment_month,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03
expense_type,,,,,,
Grt Aid to NDPBs,"£906,479,496","£1,037,462,000","£739,438,806","£717,237,137","£932,426,000","£1,172,598,913"
TOCOpCosts(Pub),"£377,832,147","£593,185,306","£1,647,041,219","£597,720,289","£706,249,531","£922,246,343"
Cap Grt Loc Auth,"£733,137,519","£53,998,796","£17,503,319","£759,745,429","£12,494,192","£422,193,732"
CurrGrtOth to NDPBs,"£786,851,304","£272,235,741","£17,900,632","£12,776,012","£561,040,873","£36,748,427"
Cap Grt NDPBs & CG,"£130,209,800","£147,991,208","£132,814,940","£124,200,000","£112,667,779","£309,494,656"


In [7]:
variance_sql = """
WITH monthly AS (
    SELECT
        payment_month,
        SUM(payment_amount_gbp) AS published_payments_gbp
    FROM payments
    GROUP BY payment_month
),
comparison AS (
    SELECT
        payment_month,
        published_payments_gbp,
        LAG(published_payments_gbp) OVER (
            ORDER BY payment_month
        ) AS previous_month_gbp
    FROM monthly
)
SELECT
    payment_month,
    ROUND(published_payments_gbp, 2) AS published_payments_gbp,
    ROUND(
        published_payments_gbp - previous_month_gbp, 2
    ) AS change_gbp,
    ROUND(
        100.0 * (published_payments_gbp - previous_month_gbp)
        / NULLIF(previous_month_gbp, 0), 2
    ) AS change_pct
FROM comparison
ORDER BY payment_month;
"""

monthly_changes = pd.read_sql_query(variance_sql, conn)

display(
    monthly_changes.style.format({
        "published_payments_gbp": "£{:,.2f}",
        "change_gbp": "£{:,.2f}",
        "change_pct": "{:+.2f}%"
    }, na_rep="—")
)

,payment_month,published_payments_gbp,change_gbp,change_pct
0,2025-10,"£3,123,556,273.58",—,—
1,2025-11,"£2,366,997,852.94","£-756,558,420.64",-24.22%
2,2025-12,"£2,900,228,687.01","£533,230,834.07",+22.53%
3,2026-01,"£2,391,570,289.87","£-508,658,397.14",-17.54%
4,2026-02,"£2,624,735,301.42","£233,165,011.55",+9.75%
5,2026-03,"£3,202,766,377.35","£578,031,075.93",+22.02%


In [8]:
march_drivers_sql = """
WITH category_changes AS (
    SELECT
        expense_type,
        SUM(CASE WHEN payment_month = '2026-02'
            THEN payment_amount_gbp ELSE 0 END) AS february_gbp,
        SUM(CASE WHEN payment_month = '2026-03'
            THEN payment_amount_gbp ELSE 0 END) AS march_gbp
    FROM payments
    WHERE payment_month IN ('2026-02', '2026-03')
    GROUP BY expense_type
)
SELECT
    expense_type,
    ROUND(february_gbp, 2) AS february_gbp,
    ROUND(march_gbp, 2) AS march_gbp,
    ROUND(march_gbp - february_gbp, 2) AS change_gbp
FROM category_changes
ORDER BY ABS(march_gbp - february_gbp) DESC;
"""

march_drivers = pd.read_sql_query(march_drivers_sql, conn)

display(
    march_drivers.head(10).style.format({
        "february_gbp": "£{:,.2f}",
        "march_gbp": "£{:,.2f}",
        "change_gbp": "£{:,.2f}"
    })
)

print(
    "Net change across all categories: "
    f"£{march_drivers['change_gbp'].sum():,.2f}"
)

,expense_type,february_gbp,march_gbp,change_gbp
0,CurrGrtOth to NDPBs,"£561,040,873.04","£36,748,427.24","£-524,292,445.80"
1,Cap Grt Loc Auth,"£12,494,191.96","£422,193,732.44","£409,699,540.48"
2,Grt Aid to NDPBs,"£932,426,000.00","£1,172,598,913.00","£240,172,913.00"
3,TOCOpCosts(Pub),"£706,249,531.35","£922,246,343.23","£215,996,811.88"
4,Cap Grt NDPBs & CG,"£112,667,779.21","£309,494,655.62","£196,826,876.41"
5,Curr Grt Other CG Bo,£0.00,"£81,668,517.25","£81,668,517.25"
6,TOCPerfMgmtFee(Pub),"£53,156,445.86",£0.00,"£-53,156,445.86"
7,Subsidies Private Se,"£49,015,405.33","£26,412,044.07","£-22,603,361.26"
8,Pte Cost Of Servs Ch,"£12,959,001.28","£25,294,325.21","£12,335,323.93"
9,TOCRevenue(Pub),"£13,066,289.03","£23,627,365.83","£10,561,076.80"


Net change across all categories: £578,031,075.93


In [9]:
march_payments_sql = """
SELECT
    payment_date,
    supplier,
    expense_type,
    item_text,
    transaction_no,
    payment_amount_gbp,
    identical_record_flag
FROM payments
WHERE payment_month = '2026-03'
ORDER BY payment_amount_gbp DESC
LIMIT 10;
"""

march_payments = pd.read_sql_query(march_payments_sql, conn)

display(
    march_payments.style.format({
        "payment_amount_gbp": "£{:,.2f}"
    })
)

,payment_date,supplier,expense_type,item_text,transaction_no,payment_amount_gbp,identical_record_flag
0,2026-03-03 00:00:00,Network Rail,Grt Aid to NDPBs,#,2000032412,"£487,671,000.00",0
1,2026-03-31 00:00:00,National Highways Limited,Grt Aid to NDPBs,Grant in Aid NH for April 2026,2000035534,"£400,000,000.00",0
2,2026-03-31 00:00:00,Network Rail,Grt Aid to NDPBs,9001331618,2000035511,"£277,465,000.00",0
3,2026-03-30 00:00:00,GOVIA THAMESLINK RAILWAY LIMITED,TOCOpCosts(Pub),2701 Wk1 GTR_NRC_PCP_Opex_(ICP) (Payment on accoun,2000035427,"£176,588,850.74",0
4,2026-03-31 00:00:00,Network Rail,Cap Grt NDPBs & CG,9001331618,2000035511,"£150,800,000.00",0
5,2026-03-30 00:00:00,GREAT WESTERN RAILWAY,TOCOpCosts(Pub),"""2701 Wk1 GWR_NRC_ICP_OPEX_P2701 """"On Account"""" FI",2000035412,"£143,802,923.09",0
6,2026-03-03 00:00:00,Network Rail,Cap Grt NDPBs & CG,#,2000032412,"£128,800,000.00",0
7,2026-03-30 00:00:00,NORTHERN TRAINS LIMITED,TOCOpCosts(Pub),2701 Wk1 NTL P2701 CCrp Cost Component Payment Nor,2000035444,"£106,396,000.00",0
8,2026-03-30 00:00:00,LONDON NORTH EASTERN RAILWAY,TOCOpCosts(Pub),2701 Wk1 LNER P2701 CCrp Cost Budget Payment Londo,2000035439,"£91,344,990.00",0
9,2026-03-30 00:00:00,FIRST TRENITALIA RAIL,TOCOpCosts(Pub),2701 Wk1 P2701 AWC NRC Contract Payment (CP) - Ope,2000035443,"£88,401,120.73",0


In [10]:
large_payment_sql = """
WITH category_stats AS (
    SELECT
        expense_type,
        COUNT(*) AS category_records,
        AVG(payment_amount_gbp) AS category_average_gbp
    FROM payments
    GROUP BY expense_type
)
SELECT
    p.payment_month,
    p.supplier,
    p.expense_type,
    p.transaction_no,
    p.payment_amount_gbp,
    c.category_records,
    ROUND(c.category_average_gbp, 2) AS category_average_gbp,
    ROUND(
        p.payment_amount_gbp / c.category_average_gbp, 2
    ) AS times_category_average
FROM payments AS p
JOIN category_stats AS c
    ON p.expense_type = c.expense_type
WHERE c.category_records >= 10
  AND p.payment_amount_gbp > 5 * c.category_average_gbp
ORDER BY times_category_average DESC;
"""

large_payment_flags = pd.read_sql_query(
    large_payment_sql, conn
)

print("Payment records flagged:", len(large_payment_flags))

display(
    large_payment_flags.head(10).style.format({
        "payment_amount_gbp": "£{:,.2f}",
        "category_average_gbp": "£{:,.2f}",
        "times_category_average": "{:.2f}"
    })
)

Payment records flagged: 98


,payment_month,supplier,expense_type,transaction_no,payment_amount_gbp,category_records,category_average_gbp,times_category_average
0,2025-11,INNOVATE UK,Research,2000022744,"£16,780,394.35",214,"£225,419.34",74.44
1,2026-02,North East Combined Authority c/o,CurrGrt Las in AEF/A,2000031995,"£23,762,653.00",198,"£713,464.57",33.31
2,2026-03,WEST MIDLANDS TRAVEL LTD,Subsidies Private Se,2000035314,"£4,294,124.12",944,"£196,095.59",21.90
3,2026-03,East Midlands Combined County Autho,CurrGrt Las in AEF/A,2000034797,"£14,507,000.00",198,"£713,464.57",20.33
4,2025-10,West Midlands Combined Authority,Cap Grt Loc Auth,2000019088,"£65,369,250.00",619,"£3,229,520.17",20.24
5,2025-10,The Commissioners of Northern,Support Services,2000018926,"£5,300,000.00",182,"£264,093.83",20.07
6,2026-02,West Midlands Combined Authority,CurrGrt Las in AEF/A,2000031104,"£12,750,000.00",198,"£713,464.57",17.87
7,2026-03,WEST OF ENGLAND COMBINED,CurrGrt Las in AEF/A,2000035277,"£12,276,000.00",198,"£713,464.57",17.21
8,2026-02,INNOVATE UK,Research,2000031467,"£3,852,748.56",214,"£225,419.34",17.09
9,2026-01,West Midlands Combined Authority,Cap Grt Loc Auth,2000027799,"£54,369,250.00",619,"£3,229,520.17",16.84


In [11]:
review_sql = """
WITH category_stats AS (
    SELECT
        expense_type,
        COUNT(*) AS category_records,
        AVG(payment_amount_gbp) AS category_average_gbp
    FROM payments
    GROUP BY expense_type
)
SELECT
    p.*,
    ROUND(
        p.payment_amount_gbp / c.category_average_gbp, 2
    ) AS times_category_average
FROM payments AS p
JOIN category_stats AS c
    ON p.expense_type = c.expense_type
WHERE c.category_records >= 10
  AND p.payment_amount_gbp > 5 * c.category_average_gbp
ORDER BY times_category_average DESC;
"""

payment_review = pd.read_sql_query(review_sql, conn)

display(
    payment_review[[
        "supplier",
        "item_text",
        "payment_amount_gbp",
        "times_category_average",
        "source_file",
        "source_row"
    ]].head(5)
)

,supplier,item_text,payment_amount_gbp,times_category_average,source_file,source_row
0,INNOVATE UK,#,16780394.35,74.44,dft_spending_2025_11.csv,23
1,North East Combined Authority c/o,25/26 North East Combined Authority,23762653.00,33.31,dft_spending_2026_02.csv,18
2,WEST MIDLANDS TRAVEL LTD,West Midlands Travel Ltd,4294124.12,21.90,dft_spending_2026_03.csv,96
3,East Midlands Combined County Autho,CRSTS/TCR Revenue,14507000.00,20.33,dft_spending_2026_03.csv,39
4,West Midlands Combined Authority,CRSTS Capital,65369250.00,20.24,dft_spending_2025_10.csv,11


In [12]:
SQL_DIR = PROJECT_ROOT / "sql"

queries = {
    "01_monthly_spending.sql": monthly_sql,
    "02_top_payment_recipients.sql": supplier_sql,
    "03_top_expense_categories.sql": category_sql,
    "04_monthly_category_spending.sql": category_monthly_sql,
    "05_monthly_payment_changes.sql": variance_sql,
    "06_march_category_drivers.sql": march_drivers_sql,
    "07_largest_march_payments.sql": march_payments_sql,
    "08_large_payment_flags.sql": large_payment_sql,
    "09_payment_review_details.sql": review_sql,
}

for filename, query in queries.items():
    (SQL_DIR / filename).write_text(
        query.strip() + "\n",
        encoding="utf-8"
    )

print("SQL queries saved:", len(queries))

SQL queries saved: 9


In [13]:
TABLEAU_DIR = PROJECT_ROOT / "tableau"

exports = {
    "monthly_spending.csv": sql_monthly_summary,
    "monthly_payment_changes.csv": monthly_changes,
    "top_10_payment_recipients.csv": top_suppliers,
    "top_10_expense_categories.csv": top_categories,
    "monthly_category_spending.csv": category_monthly,
    "march_category_drivers.csv": march_drivers,
    "large_payment_review.csv": payment_review,
}

for filename, table in exports.items():
    table.to_csv(TABLEAU_DIR / filename, index=False)

print("Tableau reporting files saved:", len(exports))

Tableau reporting files saved: 7
